<a href="https://colab.research.google.com/github/eshwar-7419/cads/blob/main/CARC_IDS_Phase2B_v2_Corrected_Continual_Learning_Benchmark_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2B-v2 — Corrected Continual Learning Benchmark

## Purpose

This is the corrected Phase-2B experiment.

The previous pilot exposed a protocol flaw: the chronological first 70% of Segment 3 could be highly class-imbalanced or effectively one-class, causing the static and naive baselines to fail before continual learning was meaningfully tested.

This version fixes that problem.

### Evidence-based stream

- **E1 — Initial attack regime:** Segment 3
- **E2 — High-attack regime:** Segments 4–6
- **E3 — Attack-family transition:** Segment 7
- **E4 — Generic-dominant regime:** Segment 8
- **E5 — Stable late regime:** Segments 9–10

Segments 1–2 are not used to initialize the supervised detector because they contain no attack examples.

### E1 protocol

E1 is split **stratified by label**:

- 56% → initial model training
- 14% → threshold/calibration set
- 30% → E1 evaluation

This preserves the E1 class distribution while preventing the initial detector from becoming a one-class model.

### Important limitation

This means E1 is not a strictly chronological micro-stream. The **continual sequence between E1, E2, E3, E4 and E5 remains sequential**, while E1 initialization is made statistically valid.

We do not claim that this split itself models real-time arrival order.

### Models

1. Static LightGBM
2. Naive continual LightGBM
3. Replay LightGBM
4. EWC MLP

EWC is implemented with an MLP because parameter-level EWC is naturally defined for differentiable neural-network parameters. We do not call it EWC-LightGBM.

### Decision threshold

Each method receives the same threshold-selection protocol:

1. train using E1-train;
2. select one threshold using E1-calibration only;
3. freeze that threshold for the entire stream and final test.

The threshold is chosen as the best F1 among thresholds satisfying an FPR constraint of at most 10% on the calibration set. If no threshold satisfies the constraint, the threshold with the lowest calibration FPR is selected, with F1 as the tie-breaker.

No later experience or final test is used to tune thresholds.

### Final test

The supplied temporal test set is completely excluded from continual-learning task construction and threshold tuning.


In [1]:
# 1. Install dependencies
!pip -q install datasets lightgbm psutil joblib scikit-learn torch

print("Dependencies installed.")


Dependencies installed.


In [2]:
# 2. Imports and reproducibility

import os
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import joblib

from datasets import load_dataset

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE = Path("/content/carc_ids_phase2b_v2")
RESULTS = BASE / "results"
ARTIFACTS = BASE / "artifacts"
RESULTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Working directory:", BASE)


Device: cpu
Working directory: /content/carc_ids_phase2b_v2


## 3. Load the exact dataset used in the earlier phases

Dataset:

`lacg030175/UNSW-NB15`

Configuration:

`temporal`

Only the training split is used to build the continual stream.


In [3]:
# 3. Load dataset

ds = load_dataset(
    "lacg030175/UNSW-NB15",
    "temporal"
)

train_df = ds["train"].to_pandas()
test_df = ds["test"].to_pandas()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTrain labels:")
display(train_df["label"].value_counts(dropna=False).sort_index().rename("count").to_frame())

print("\nTest labels:")
display(test_df["label"].value_counts(dropna=False).sort_index().rename("count").to_frame())


README.md:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

temporal/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.7MB            

temporal/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

temporal/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.25MB            

temporal/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating test split:   0%|          | 0/82332 [00:00<?, ? examples/s]

Train: (175341, 44)
Test : (82332, 44)

Train labels:


,count
label,
0,56000
1,119341



Test labels:


,count
label,
0,37000
1,45332


# 4. Reconstruct the evidence-based sequential stream

We retain the boundaries justified by Phase-2A:

| Experience | Segments | Purpose |
|---|---|---|
| E1 | 3 | Initial attack-aware detector |
| E2 | 4–6 | High-attack regime |
| E3 | 7 | Attack-family transition |
| E4 | 8 | Generic-dominant regime |
| E5 | 9–10 | Stable late regime |

We do not create artificial tasks for every segment.


In [4]:
# 4. Ten sequential segments

N_SEGMENTS = 10
segment_indices = np.array_split(np.arange(len(train_df)), N_SEGMENTS)

segments = {
    sid: train_df.iloc[idx].copy()
    for sid, idx in enumerate(segment_indices, start=1)
}

EXPERIENCES = {
    "E1_initial_attack": [3],
    "E2_high_attack": [4, 5, 6],
    "E3_attack_transition": [7],
    "E4_generic_dominant": [8],
    "E5_stable_late": [9, 10]
}

experience_dfs = {
    name: pd.concat([segments[s] for s in sids], ignore_index=True)
    for name, sids in EXPERIENCES.items()
}

summary_rows = []

for name, df in experience_dfs.items():
    summary_rows.append({
        "experience": name,
        "segments": ",".join(map(str, EXPERIENCES[name])),
        "rows": len(df),
        "benign": int((df.label == 0).sum()),
        "attack": int((df.label == 1).sum()),
        "attack_pct": float(df.label.mean()),
        "attack_categories": int(df.loc[df.label == 1, "attack_cat"].nunique())
    })

experience_summary = pd.DataFrame(summary_rows)
display(experience_summary)

experience_summary.to_csv(
    RESULTS / "experience_summary.csv",
    index=False
)


,experience,segments,rows,benign,attack,attack_pct,attack_categories
0,E1_initial_attack,3,17534,12842,4692,0.267594,8
1,E2_high_attack,"4,5,6",52602,5405,47197,0.897247,8
2,E3_attack_transition,7,17534,2684,14850,0.846926,9
3,E4_generic_dominant,8,17534,0,17534,1.000000,9
4,E5_stable_late,"9,10",35068,0,35068,1.000000,9


# 5. Leakage-safe feature representation

We use the original feature representation.

Removed:

- `label`
- `attack_cat`
- obvious identifier columns

The preprocessing transformer is fitted on the complete supplied **training split only**.

The temporal test set is never used to fit preprocessing.


In [5]:
# 5. Prepare raw features

DROP = [
    c for c in ["label", "attack_cat", "id", "ID", "index"]
    if c in train_df.columns
]

X_train_raw = train_df.drop(columns=DROP, errors="ignore").copy()
X_test_raw = test_df.drop(columns=DROP, errors="ignore").copy()

y_train = train_df["label"].astype(int).to_numpy()
y_test = test_df["label"].astype(int).to_numpy()

numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_train_raw.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_cols)
])

X_train_proc = preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_test_proc = preprocessor.transform(X_test_raw).astype(np.float32)

joblib.dump(preprocessor, ARTIFACTS / "preprocessor.joblib")

print("Processed train:", X_train_proc.shape)
print("Processed test :", X_test_proc.shape)


Processed train: (175341, 194)
Processed test : (82332, 194)


# 6. Correct E1 initialization split

This is the key correction.

E1 is stratified:

- 56% model training
- 14% calibration
- 30% evaluation

The calibration set is used only for threshold selection.

The evaluation set is untouched until E1 evaluation.

The remaining experiences are split chronologically into adaptation/evaluation portions.


In [6]:
# 6. E1 stratified split

e1_df = experience_dfs["E1_initial_attack"].copy()

e1_train_df, e1_holdout_df = train_test_split(
    e1_df,
    test_size=0.44,
    stratify=e1_df["label"],
    random_state=SEED
)

e1_cal_df, e1_eval_df = train_test_split(
    e1_holdout_df,
    test_size=(0.30 / 0.44),
    stratify=e1_holdout_df["label"],
    random_state=SEED
)

print("E1 total:", len(e1_df))
print("E1 train:", len(e1_train_df))
print("E1 calibration:", len(e1_cal_df))
print("E1 evaluation:", len(e1_eval_df))

print("\nClass distributions:")
for name, df in [
    ("E1 total", e1_df),
    ("E1 train", e1_train_df),
    ("E1 calibration", e1_cal_df),
    ("E1 evaluation", e1_eval_df)
]:
    print(
        name,
        "benign =", int((df.label == 0).sum()),
        "attack =", int((df.label == 1).sum()),
        "attack_pct =", round(float(df.label.mean()), 4)
    )

assert e1_train_df["label"].nunique() == 2
assert e1_cal_df["label"].nunique() == 2
assert e1_eval_df["label"].nunique() == 2


E1 total: 17534
E1 train: 9819
E1 calibration: 2454
E1 evaluation: 5261

Class distributions:
E1 total benign = 12842 attack = 4692 attack_pct = 0.2676
E1 train benign = 7191 attack = 2628 attack_pct = 0.2676
E1 calibration benign = 1797 attack = 657 attack_pct = 0.2677
E1 evaluation benign = 3854 attack = 1407 attack_pct = 0.2674


In [7]:
# 7. Transform E1 partitions

def transform_df(df):
    X = df.drop(columns=DROP, errors="ignore").copy()
    y = df["label"].astype(int).to_numpy()
    Xp = preprocessor.transform(X).astype(np.float32)
    return Xp, y

E1_X_train, E1_y_train = transform_df(e1_train_df)
E1_X_cal, E1_y_cal = transform_df(e1_cal_df)
E1_X_eval, E1_y_eval = transform_df(e1_eval_df)

print(E1_X_train.shape, E1_y_train.shape)
print(E1_X_cal.shape, E1_y_cal.shape)
print(E1_X_eval.shape, E1_y_eval.shape)


(9819, 194) (9819,)
(2454, 194) (2454,)
(5261, 194) (5261,)


In [8]:
# 8. Build E2-E5 chronological adaptation/evaluation partitions

stream_data = {
    "E1_initial_attack": {
        "X_adapt": E1_X_train,
        "y_adapt": E1_y_train,
        "X_cal": E1_X_cal,
        "y_cal": E1_y_cal,
        "X_eval": E1_X_eval,
        "y_eval": E1_y_eval
    }
}

for name in list(EXPERIENCES.keys())[1:]:
    df = experience_dfs[name].copy()

    split = int(len(df) * 0.70)
    split = max(1, min(split, len(df)-1))

    adapt_df = df.iloc[:split].copy()
    eval_df = df.iloc[split:].copy()

    X_adapt, y_adapt = transform_df(adapt_df)
    X_eval, y_eval = transform_df(eval_df)

    stream_data[name] = {
        "X_adapt": X_adapt,
        "y_adapt": y_adapt,
        "X_eval": X_eval,
        "y_eval": y_eval
    }

    print(
        name,
        "adapt:", X_adapt.shape,
        "eval:", X_eval.shape,
        "adapt attack rate:", round(float(y_adapt.mean()), 4),
        "eval attack rate:", round(float(y_eval.mean()), 4)
    )


E2_high_attack adapt: (36821, 194) eval: (15781, 194) adapt attack rate: 0.8986 eval attack rate: 0.8941
E3_attack_transition adapt: (12273, 194) eval: (5261, 194) adapt attack rate: 0.7813 eval attack rate: 1.0
E4_generic_dominant adapt: (12273, 194) eval: (5261, 194) adapt attack rate: 1.0 eval attack rate: 1.0
E5_stable_late adapt: (24547, 194) eval: (10521, 194) adapt attack rate: 1.0 eval attack rate: 1.0


# 9. Common metric and threshold protocol

A threshold is not fixed at 0.50.

That would unfairly penalize models with different probability calibration.

Instead, each model uses the same **selection rule** on the same E1 calibration set:

1. evaluate thresholds from 0.05 to 0.95;
2. identify thresholds with calibration FPR <= 10%;
3. choose the threshold with maximum F1;
4. if none meets the constraint, choose the threshold with minimum FPR, then maximum F1;
5. freeze that threshold for all later experiences and the final test.

This means threshold tuning never uses E2-E5 or the final test.


In [9]:
# 9. Metrics and threshold selection

def binary_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "roc_auc": roc_auc_score(y_true, probs),
        "pr_auc": average_precision_score(y_true, probs),
        "tp": int(tp), "fp": int(fp),
        "tn": int(tn), "fn": int(fn)
    }

def select_threshold(y_true, probs, fpr_limit=0.10):
    rows = []

    for threshold in np.linspace(0.05, 0.95, 181):
        m = binary_metrics(y_true, probs, threshold)
        rows.append({
            "threshold": threshold,
            "f1": m["f1"],
            "recall": m["recall"],
            "precision": m["precision"],
            "fpr": m["fpr"]
        })

    table = pd.DataFrame(rows)

    feasible = table[table["fpr"] <= fpr_limit]

    if len(feasible):
        best = feasible.sort_values(
            ["f1", "recall", "fpr"],
            ascending=[False, False, True]
        ).iloc[0]
        reason = "best_f1_under_fpr_constraint"
    else:
        best = table.sort_values(
            ["fpr", "f1"],
            ascending=[True, False]
        ).iloc[0]
        reason = "minimum_fpr_fallback"

    return float(best["threshold"]), table, reason


# 10. Resource measurement helpers


In [10]:
# 10. Resource helpers

def rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

def resource_snapshot():
    return {
        "rss_mb": rss_mb(),
        "cpu_percent": psutil.cpu_percent(interval=0.1)
    }


# 11. Static LightGBM

Static training:

- E1 train only

Threshold:

- E1 calibration only

Then:

- no updates during E2-E5.

This is the clean non-adaptive baseline.


In [11]:
# 11. Train static LightGBM

static_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

t0 = time.perf_counter()
static_model.fit(E1_X_train, E1_y_train)
static_train_time = time.perf_counter() - t0

static_cal_probs = static_model.predict_proba(E1_X_cal)[:, 1]

static_threshold, static_threshold_table, static_reason = select_threshold(
    E1_y_cal,
    static_cal_probs
)

static_threshold_table.to_csv(
    RESULTS / "static_threshold_selection.csv",
    index=False
)

print("Static threshold:", static_threshold)
print("Reason:", static_reason)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Static threshold: 0.4499999999999999
Reason: best_f1_under_fpr_constraint


In [12]:
# 12. Evaluate static model across stream

static_rows = []

for exp_name, data in stream_data.items():
    probs = static_model.predict_proba(data["X_eval"])[:,1]

    m = binary_metrics(
        data["y_eval"],
        probs,
        static_threshold
    )

    m.update({
        "method": "Static_LightGBM",
        "model_state": "E1",
        "evaluated_experience": exp_name,
        "threshold": static_threshold,
        "train_time_sec": static_train_time
    })

    static_rows.append(m)

static_stream = pd.DataFrame(static_rows)

display(
    static_stream[
        ["method","evaluated_experience","threshold",
         "f1","recall","precision","fpr","roc_auc","pr_auc"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklear

,method,evaluated_experience,threshold,f1,recall,precision,fpr,roc_auc,pr_auc
0,Static_LightGBM,E1_initial_attack,0.45,0.8712,0.8436,0.9006,0.0340,0.9787,0.9538
1,Static_LightGBM,E2_high_attack,0.45,0.8846,0.8209,0.9589,0.2968,0.8768,0.9827
2,Static_LightGBM,E3_attack_transition,0.45,0.8774,0.7816,1.0000,NaN,NaN,1.0000
3,Static_LightGBM,E4_generic_dominant,0.45,0.8509,0.7405,1.0000,NaN,NaN,1.0000
4,Static_LightGBM,E5_stable_late,0.45,0.8581,0.7514,1.0000,NaN,NaN,1.0000


# 13. Naive continual LightGBM

At every experience, retrain on only the current experience's adaptation data.

The threshold remains frozen at the E1-calibrated threshold.

This tests whether simple adaptation improves current-regime performance at the expense of retention.


In [13]:
# 13. Naive continual LightGBM

naive_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

naive_rows = []
naive_threshold = None

for i, (exp_name, data) in enumerate(stream_data.items(), start=1):

    t0 = time.perf_counter()
    naive_model.fit(data["X_adapt"], data["y_adapt"])
    train_time = time.perf_counter() - t0

    if i == 1:
        cal_probs = naive_model.predict_proba(data["X_cal"])[:,1]
        naive_threshold, naive_threshold_table, naive_reason = select_threshold(
            data["y_cal"], cal_probs
        )
        naive_threshold_table.to_csv(
            RESULTS / "naive_threshold_selection.csv",
            index=False
        )
        print("Naive threshold:", naive_threshold)

    for eval_name, eval_data in list(stream_data.items())[:i]:
        probs = naive_model.predict_proba(eval_data["X_eval"])[:,1]

        m = binary_metrics(
            eval_data["y_eval"],
            probs,
            naive_threshold
        )

        m.update({
            "method": "Naive_CL_LightGBM",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "threshold": naive_threshold,
            "train_time_sec": train_time,
            "rss_mb": rss_mb()
        })

        naive_rows.append(m)

naive_stream = pd.DataFrame(naive_rows)

display(
    naive_stream[
        ["method","after_experience","evaluated_experience",
         "f1","recall","precision","fpr"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Naive threshold: 0.4499999999999999


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,Naive_CL_LightGBM,E1_initial_attack,E1_initial_attack,0.8712,0.8436,0.9006,0.0340
1,Naive_CL_LightGBM,E2_high_attack,E1_initial_attack,0.4441,0.9524,0.2896,0.8529
2,Naive_CL_LightGBM,E2_high_attack,E2_high_attack,0.9551,0.9851,0.9269,0.6559
3,Naive_CL_LightGBM,E3_attack_transition,E1_initial_attack,0.4305,0.9758,0.2761,0.9338
4,Naive_CL_LightGBM,E3_attack_transition,E2_high_attack,0.9427,0.9890,0.9006,0.9222
5,Naive_CL_LightGBM,E3_attack_transition,E3_attack_transition,0.9968,0.9935,1.0000,NaN
6,Naive_CL_LightGBM,E4_generic_dominant,E1_initial_attack,0.0000,0.0000,0.0000,0.0000
7,Naive_CL_LightGBM,E4_generic_dominant,E2_high_attack,0.0000,0.0000,0.0000,0.0000
8,Naive_CL_LightGBM,E4_generic_dominant,E3_attack_transition,0.0000,0.0000,0.0000,NaN
9,Naive_CL_LightGBM,E4_generic_dominant,E4_generic_dominant,0.0000,0.0000,0.0000,NaN


# 14. Replay LightGBM

Replay maintains a bounded historical memory.

At each experience:

\[
D_t^{fit}=D_t^{current}\cup R_t
\]

where `R_t` is bounded replay memory.

The replay budget is intentionally explicit and measurable.


In [14]:
# 14. Replay continual learning

REPLAY_PER_EXPERIENCE = 2000
rng = np.random.default_rng(SEED)

replay_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)

replay_X = []
replay_y = []
replay_rows = []
replay_threshold = None

for i, (exp_name, data) in enumerate(stream_data.items(), start=1):

    if replay_X:
        X_old = np.concatenate(replay_X, axis=0)
        y_old = np.concatenate(replay_y, axis=0)

        X_fit = np.concatenate([data["X_adapt"], X_old], axis=0)
        y_fit = np.concatenate([data["y_adapt"], y_old], axis=0)
    else:
        X_fit = data["X_adapt"]
        y_fit = data["y_adapt"]

    t0 = time.perf_counter()
    replay_model.fit(X_fit, y_fit)
    train_time = time.perf_counter() - t0

    if i == 1:
        cal_probs = replay_model.predict_proba(data["X_cal"])[:,1]
        replay_threshold, replay_threshold_table, replay_reason = select_threshold(
            data["y_cal"], cal_probs
        )
        replay_threshold_table.to_csv(
            RESULTS / "replay_threshold_selection.csv",
            index=False
        )
        print("Replay threshold:", replay_threshold)

    for eval_name, eval_data in list(stream_data.items())[:i]:
        probs = replay_model.predict_proba(eval_data["X_eval"])[:,1]

        m = binary_metrics(
            eval_data["y_eval"],
            probs,
            replay_threshold
        )

        m.update({
            "method": "Replay_LightGBM",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "threshold": replay_threshold,
            "train_time_sec": train_time,
            "replay_size": sum(len(x) for x in replay_X),
            "rss_mb": rss_mb()
        })

        replay_rows.append(m)

    n = min(REPLAY_PER_EXPERIENCE, len(data["X_adapt"]))
    chosen = rng.choice(len(data["X_adapt"]), size=n, replace=False)

    replay_X.append(data["X_adapt"][chosen])
    replay_y.append(data["y_adapt"][chosen])

replay_stream = pd.DataFrame(replay_rows)

display(
    replay_stream[
        ["method","after_experience","evaluated_experience",
         "f1","recall","precision","fpr","replay_size"]
    ].round(4)
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Replay threshold: 0.4499999999999999


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,method,after_experience,evaluated_experience,f1,recall,precision,fpr,replay_size
0,Replay_LightGBM,E1_initial_attack,E1_initial_attack,0.8712,0.8436,0.9006,0.0340,0
1,Replay_LightGBM,E2_high_attack,E1_initial_attack,0.6776,0.9574,0.5243,0.3171,2000
2,Replay_LightGBM,E2_high_attack,E2_high_attack,0.9694,0.9807,0.9583,0.3603,2000
3,Replay_LightGBM,E3_attack_transition,E1_initial_attack,0.7283,0.9431,0.5932,0.2361,4000
4,Replay_LightGBM,E3_attack_transition,E2_high_attack,0.9639,0.9697,0.9582,0.3573,4000
5,Replay_LightGBM,E3_attack_transition,E3_attack_transition,0.9917,0.9835,1.0000,NaN,4000
6,Replay_LightGBM,E4_generic_dominant,E1_initial_attack,0.7160,0.9595,0.5711,0.2631,6000
7,Replay_LightGBM,E4_generic_dominant,E2_high_attack,0.9634,0.9732,0.9538,0.3980,6000
8,Replay_LightGBM,E4_generic_dominant,E3_attack_transition,0.9952,0.9905,1.0000,NaN,6000
9,Replay_LightGBM,E4_generic_dominant,E4_generic_dominant,0.9969,0.9937,1.0000,NaN,6000


# 15. Compact EWC MLP

EWC is implemented as a neural-network baseline.

The same E1 train/calibration protocol is used.

The threshold is then frozen.

EWC objective:

\[
L=L_{current}+
\frac{\lambda}{2}
\sum_iF_i(\theta_i-\theta_i^*)^2
\]



In [15]:
# 15. MLP and EWC helpers

class SmallMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden = min(128, max(32, input_dim // 4))
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

def make_loader(X, y, batch_size=256, shuffle=True):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_ewc(model, X, y, old_params=None, fisher=None,
              ewc_lambda=100.0, epochs=5, lr=1e-3):

    model.train()
    loader = make_loader(X, y)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    t0 = time.perf_counter()

    for _ in range(epochs):
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)

            if old_params is not None and fisher is not None:
                penalty = 0.0
                for name, param in model.named_parameters():
                    penalty += (
                        fisher[name] *
                        (param - old_params[name])**2
                    ).sum()

                loss = loss + (ewc_lambda / 2.0) * penalty

            loss.backward()
            optimizer.step()

    return time.perf_counter() - t0

@torch.no_grad()
def mlp_probs(model, X):
    model.eval()

    loader = make_loader(
        X,
        np.zeros(len(X), dtype=np.float32),
        shuffle=False
    )

    out = []

    for xb, _ in loader:
        xb = xb.to(DEVICE)
        out.append(
            torch.sigmoid(model(xb)).cpu().numpy()
        )

    return np.concatenate(out)

def estimate_fisher(model, X, y):
    model.eval()
    loader = make_loader(X, y)
    criterion = nn.BCEWithLogitsLoss()

    fisher = {
        name: torch.zeros_like(param, device=DEVICE)
        for name, param in model.named_parameters()
    }

    total = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        model.zero_grad()

        loss = criterion(model(xb), yb)
        loss.backward()

        n = len(xb)
        total += n

        for name, param in model.named_parameters():
            if param.grad is not None:
                fisher[name] += (
                    param.grad.detach()**2
                ) * n

    for name in fisher:
        fisher[name] /= max(total, 1)

    return fisher


In [16]:
# 16. EWC experiment

ewc_model = SmallMLP(
    stream_data["E1_initial_attack"]["X_adapt"].shape[1]
).to(DEVICE)

EWC_LAMBDA = 100.0
EWC_EPOCHS = 5

old_params = None
fisher = None

ewc_rows = []
ewc_threshold = None

for i, (exp_name, data) in enumerate(stream_data.items(), start=1):

    resource_before = resource_snapshot()

    train_time = train_ewc(
        ewc_model,
        data["X_adapt"],
        data["y_adapt"],
        old_params=old_params,
        fisher=fisher,
        ewc_lambda=EWC_LAMBDA,
        epochs=EWC_EPOCHS,
        lr=1e-3
    )

    resource_after = resource_snapshot()

    if i == 1:
        cal_probs = mlp_probs(
            ewc_model,
            data["X_cal"]
        )

        ewc_threshold, ewc_threshold_table, ewc_reason = select_threshold(
            data["y_cal"],
            cal_probs
        )

        ewc_threshold_table.to_csv(
            RESULTS / "ewc_threshold_selection.csv",
            index=False
        )

        print("EWC threshold:", ewc_threshold)

    for eval_name, eval_data in list(stream_data.items())[:i]:

        probs = mlp_probs(
            ewc_model,
            eval_data["X_eval"]
        )

        m = binary_metrics(
            eval_data["y_eval"],
            probs,
            ewc_threshold
        )

        m.update({
            "method": "EWC_MLP",
            "after_experience": exp_name,
            "evaluated_experience": eval_name,
            "threshold": ewc_threshold,
            "train_time_sec": train_time,
            "rss_mb": resource_after["rss_mb"]
        })

        ewc_rows.append(m)

    fisher = estimate_fisher(
        ewc_model,
        data["X_adapt"],
        data["y_adapt"]
    )

    old_params = {
        name: param.detach().clone()
        for name, param in ewc_model.named_parameters()
    }

ewc_stream = pd.DataFrame(ewc_rows)

display(
    ewc_stream[
        ["method","after_experience","evaluated_experience",
         "f1","recall","precision","fpr"]
    ].round(4)
)


EWC threshold: 0.37999999999999995


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist

,method,after_experience,evaluated_experience,f1,recall,precision,fpr
0,EWC_MLP,E1_initial_attack,E1_initial_attack,0.7214,0.6930,0.7523,0.0833
1,EWC_MLP,E2_high_attack,E1_initial_attack,0.4939,0.9851,0.3296,0.7314
2,EWC_MLP,E2_high_attack,E2_high_attack,0.9638,0.9962,0.9333,0.6008
3,EWC_MLP,E3_attack_transition,E1_initial_attack,0.4856,1.0000,0.3206,0.7735
4,EWC_MLP,E3_attack_transition,E2_high_attack,0.9503,1.0000,0.9052,0.8839
5,EWC_MLP,E3_attack_transition,E3_attack_transition,1.0000,1.0000,1.0000,NaN
6,EWC_MLP,E4_generic_dominant,E1_initial_attack,0.4361,1.0000,0.2789,0.9440
7,EWC_MLP,E4_generic_dominant,E2_high_attack,0.9457,1.0000,0.8971,0.9689
8,EWC_MLP,E4_generic_dominant,E3_attack_transition,1.0000,1.0000,1.0000,NaN
9,EWC_MLP,E4_generic_dominant,E4_generic_dominant,1.0000,1.0000,1.0000,NaN


# 17. Combine continual-learning matrices

For each method we retain the full matrix:

\[
M_{i,j}
\]

where:

- `i` = model state after experience `i`;
- `j` = evaluation experience.

This is the main evidence for adaptation and forgetting.


In [17]:
# 17. Build performance matrices

def make_matrix(df, method):
    if method == "Static_LightGBM":
        # Static model is evaluated independently on all experiences.
        out = {}
        for _, row in df.iterrows():
            out[row["evaluated_experience"]] = row["f1"]
        return pd.Series(out, name=method)

    final_after_order = list(stream_data.keys())

    rows = []

    for after in final_after_order:
        g = df[df["after_experience"] == after]

        if g.empty:
            continue

        row = {"model_state": after}

        for _, r in g.iterrows():
            row[r["evaluated_experience"]] = r["f1"]

        rows.append(row)

    return pd.DataFrame(rows).set_index("model_state")

static_matrix = make_matrix(
    static_stream,
    "Static_LightGBM"
)

naive_matrix = make_matrix(
    naive_stream,
    "Naive_CL_LightGBM"
)

replay_matrix = make_matrix(
    replay_stream,
    "Replay_LightGBM"
)

ewc_matrix = make_matrix(
    ewc_stream,
    "EWC_MLP"
)

print("Static:")
display(static_matrix.round(4))

print("Naive:")
display(naive_matrix.round(4))

print("Replay:")
display(replay_matrix.round(4))

print("EWC:")
display(ewc_matrix.round(4))


Static:


,Static_LightGBM
E1_initial_attack,0.8712
E2_high_attack,0.8846
E3_attack_transition,0.8774
E4_generic_dominant,0.8509
E5_stable_late,0.8581


Naive:


,E1_initial_attack,E2_high_attack,E3_attack_transition,E4_generic_dominant,E5_stable_late
model_state,,,,,
E1_initial_attack,0.8712,NaN,NaN,NaN,NaN
E2_high_attack,0.4441,0.9551,NaN,NaN,NaN
E3_attack_transition,0.4305,0.9427,0.9968,NaN,NaN
E4_generic_dominant,0.0000,0.0000,0.0000,0.0,NaN
E5_stable_late,0.0000,0.0000,0.0000,0.0,0.0


Replay:


,E1_initial_attack,E2_high_attack,E3_attack_transition,E4_generic_dominant,E5_stable_late
model_state,,,,,
E1_initial_attack,0.8712,NaN,NaN,NaN,NaN
E2_high_attack,0.6776,0.9694,NaN,NaN,NaN
E3_attack_transition,0.7283,0.9639,0.9917,NaN,NaN
E4_generic_dominant,0.7160,0.9634,0.9952,0.9969,NaN
E5_stable_late,0.7016,0.9626,0.9962,0.9959,0.9984


EWC:


,E1_initial_attack,E2_high_attack,E3_attack_transition,E4_generic_dominant,E5_stable_late
model_state,,,,,
E1_initial_attack,0.7214,NaN,NaN,NaN,NaN
E2_high_attack,0.4939,0.9638,NaN,NaN,NaN
E3_attack_transition,0.4856,0.9503,1.0,NaN,NaN
E4_generic_dominant,0.4361,0.9457,1.0,1.0,NaN
E5_stable_late,0.4279,0.9456,1.0,1.0,1.0


In [18]:
# 18. Save matrices

naive_matrix.to_csv(RESULTS/"naive_f1_matrix.csv")
replay_matrix.to_csv(RESULTS/"replay_f1_matrix.csv")
ewc_matrix.to_csv(RESULTS/"ewc_f1_matrix.csv")

pd.DataFrame({
    "experience": list(stream_data.keys()),
    "static_f1": [
        static_stream.loc[
            static_stream["evaluated_experience"] == e, "f1"
        ].iloc[0]
        for e in stream_data
    ]
}).to_csv(
    RESULTS/"static_f1_by_experience.csv",
    index=False
)


# 19. Forgetting

For each previously learned experience:

\[
F_i =
\max_t M_{t,i} - M_{T,i}
\]

where `T` is the final model state.

This is calculated only from the continual-learning matrices.


In [19]:
# 19. Forgetting calculation

def calculate_forgetting(matrix):
    rows = []

    experiences = list(stream_data.keys())

    for exp in experiences:
        vals = matrix[exp].dropna()

        if len(vals) == 0:
            continue

        best = vals.max()
        final = matrix.iloc[-1][exp]

        rows.append({
            "experience": exp,
            "best_f1": best,
            "final_f1": final,
            "forgetting": best - final
        })

    return pd.DataFrame(rows)

forget_naive = calculate_forgetting(naive_matrix)
forget_naive["method"] = "Naive_CL_LightGBM"

forget_replay = calculate_forgetting(replay_matrix)
forget_replay["method"] = "Replay_LightGBM"

forget_ewc = calculate_forgetting(ewc_matrix)
forget_ewc["method"] = "EWC_MLP"

forgetting = pd.concat(
    [forget_naive, forget_replay, forget_ewc],
    ignore_index=True
)

display(forgetting.round(6))

forgetting.to_csv(
    RESULTS/"forgetting_analysis.csv",
    index=False
)


,experience,best_f1,final_f1,forgetting,method
0,E1_initial_attack,0.871193,0.000000,0.871193,Naive_CL_LightGBM
1,E2_high_attack,0.955130,0.000000,0.955130,Naive_CL_LightGBM
2,E3_attack_transition,0.996758,0.000000,0.996758,Naive_CL_LightGBM
3,E4_generic_dominant,0.000000,0.000000,0.000000,Naive_CL_LightGBM
4,E5_stable_late,0.000000,0.000000,0.000000,Naive_CL_LightGBM
5,E1_initial_attack,0.871193,0.701627,0.169565,Replay_LightGBM
6,E2_high_attack,0.969351,0.962584,0.006767,Replay_LightGBM
7,E3_attack_transition,0.996184,0.996184,0.000000,Replay_LightGBM
8,E4_generic_dominant,0.996854,0.995897,0.000957,Replay_LightGBM
9,E5_stable_late,0.998429,0.998429,0.000000,Replay_LightGBM


# 20. Final stream retention

After E5, evaluate the final state on every experience evaluation set.

This tells us whether the final model retains old regimes while handling the latest regime.


In [20]:
# 20. Final retention table

final_retention_rows = []

for method, matrix in [
    ("Naive_CL_LightGBM", naive_matrix),
    ("Replay_LightGBM", replay_matrix),
    ("EWC_MLP", ewc_matrix)
]:
    final_row = matrix.iloc[-1]

    for exp in stream_data.keys():
        final_retention_rows.append({
            "method": method,
            "experience": exp,
            "final_f1": final_row.get(exp, np.nan)
        })

# Static
for exp in stream_data.keys():
    static_value = static_stream.loc[
        static_stream["evaluated_experience"] == exp,
        "f1"
    ].iloc[0]

    final_retention_rows.append({
        "method": "Static_LightGBM",
        "experience": exp,
        "final_f1": static_value
    })

final_retention = pd.DataFrame(final_retention_rows)

display(final_retention.round(6))

final_retention.to_csv(
    RESULTS/"final_stream_retention.csv",
    index=False
)


,method,experience,final_f1
0,Naive_CL_LightGBM,E1_initial_attack,0.000000
1,Naive_CL_LightGBM,E2_high_attack,0.000000
2,Naive_CL_LightGBM,E3_attack_transition,0.000000
3,Naive_CL_LightGBM,E4_generic_dominant,0.000000
4,Naive_CL_LightGBM,E5_stable_late,0.000000
5,Replay_LightGBM,E1_initial_attack,0.701627
6,Replay_LightGBM,E2_high_attack,0.962584
7,Replay_LightGBM,E3_attack_transition,0.996184
8,Replay_LightGBM,E4_generic_dominant,0.995897
9,Replay_LightGBM,E5_stable_late,0.998429


# 21. Resource measurements

Resource cost is part of the research problem.

We record:

- training time;
- process RSS;
- replay memory.

The eventual proposed controller will use this evidence to determine whether adaptation should happen.


In [21]:
# 21. Resource summaries

resource_rows = []

# Static
resource_rows.append({
    "method": "Static_LightGBM",
    "total_training_time_sec": static_train_time,
    "max_rss_mb": np.nan
})

# Naive
resource_rows.append({
    "method": "Naive_CL_LightGBM",
    "total_training_time_sec": naive_stream["train_time_sec"].sum(),
    "max_rss_mb": naive_stream["rss_mb"].max()
})

# Replay
resource_rows.append({
    "method": "Replay_LightGBM",
    "total_training_time_sec": replay_stream["train_time_sec"].sum(),
    "max_rss_mb": replay_stream["rss_mb"].max(),
    "max_replay_size": replay_stream["replay_size"].max()
})

# EWC
resource_rows.append({
    "method": "EWC_MLP",
    "total_training_time_sec": ewc_stream["train_time_sec"].sum(),
    "max_rss_mb": ewc_stream["rss_mb"].max()
})

resource_summary = pd.DataFrame(resource_rows)

display(resource_summary.round(4))

resource_summary.to_csv(
    RESULTS/"resource_summary.csv",
    index=False
)


,method,total_training_time_sec,max_rss_mb,max_replay_size
0,Static_LightGBM,10.3909,NaN,NaN
1,Naive_CL_LightGBM,14.9047,1336.5977,NaN
2,Replay_LightGBM,33.7884,1359.3906,8000.0
3,EWC_MLP,37.7037,1461.0156,NaN


# 22. Final independent temporal test

The threshold is frozen from E1 calibration.

No final-test threshold tuning is permitted.

This is the final independent evaluation after the continual stream.


In [22]:
# 22. Final test evaluation

test_rows = []

# Static
probs = static_model.predict_proba(X_test_proc)[:,1]
m = binary_metrics(y_test, probs, static_threshold)
m.update({
    "method": "Static_LightGBM",
    "threshold": static_threshold
})
test_rows.append(m)

# Naive
probs = naive_model.predict_proba(X_test_proc)[:,1]
m = binary_metrics(y_test, probs, naive_threshold)
m.update({
    "method": "Naive_CL_LightGBM",
    "threshold": naive_threshold
})
test_rows.append(m)

# Replay
probs = replay_model.predict_proba(X_test_proc)[:,1]
m = binary_metrics(y_test, probs, replay_threshold)
m.update({
    "method": "Replay_LightGBM",
    "threshold": replay_threshold
})
test_rows.append(m)

# EWC
probs = mlp_probs(ewc_model, X_test_proc)
m = binary_metrics(y_test, probs, ewc_threshold)
m.update({
    "method": "EWC_MLP",
    "threshold": ewc_threshold
})
test_rows.append(m)

final_test = pd.DataFrame(test_rows)

display(
    final_test[
        ["method","threshold","accuracy","precision","recall",
         "f1","fpr","roc_auc","pr_auc"]
    ].round(6)
)

final_test.to_csv(
    RESULTS/"final_temporal_test_comparison.csv",
    index=False
)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,threshold,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc
0,Static_LightGBM,0.45,0.815236,0.906697,0.740647,0.815303,0.093378,0.935382,0.942424
1,Naive_CL_LightGBM,0.45,0.449400,0.000000,0.000000,0.000000,0.000000,0.500000,0.550600
2,Replay_LightGBM,0.45,0.841204,0.778721,0.994066,0.873314,0.346081,0.979256,0.984158
3,EWC_MLP,0.38,0.561896,0.556890,1.000000,0.715388,0.974865,0.712605,0.681011


# 23. Aggregate comparison

A continual method should not be judged by one number.

We report:

- mean final stream F1;
- mean forgetting;
- final independent-test F1;
- final independent-test FPR;
- total adaptation time.

The proposed resource-aware controller will have to improve the security/cost trade-off rather than merely maximize F1.


In [23]:
# 23. Aggregate results

aggregate = []

for method in [
    "Static_LightGBM",
    "Naive_CL_LightGBM",
    "Replay_LightGBM",
    "EWC_MLP"
]:
    g = final_retention[
        final_retention["method"] == method
    ]

    row = {
        "method": method,
        "mean_final_stream_f1": g["final_f1"].mean()
    }

    fg = forgetting[
        forgetting["method"] == method
    ]

    row["mean_forgetting"] = (
        fg["forgetting"].mean()
        if len(fg) else 0.0
    )

    test_row = final_test[
        final_test["method"] == method
    ].iloc[0]

    row["final_test_f1"] = test_row["f1"]
    row["final_test_recall"] = test_row["recall"]
    row["final_test_fpr"] = test_row["fpr"]

    r = resource_summary[
        resource_summary["method"] == method
    ].iloc[0]

    row["total_training_time_sec"] = r["total_training_time_sec"]

    aggregate.append(row)

aggregate_df = pd.DataFrame(aggregate)

display(aggregate_df.round(6))

aggregate_df.to_csv(
    RESULTS/"continual_learning_aggregate_results.csv",
    index=False
)


,method,mean_final_stream_f1,mean_forgetting,final_test_f1,final_test_recall,final_test_fpr,total_training_time_sec
0,Static_LightGBM,0.868440,0.000000,0.815303,0.740647,0.093378,10.390858
1,Naive_CL_LightGBM,0.000000,0.564616,0.000000,0.000000,0.000000,14.904721
2,Replay_LightGBM,0.930944,0.035458,0.873314,0.994066,0.346081,33.788390
3,EWC_MLP,0.874681,0.062356,0.715388,1.000000,0.974865,37.703688


# 24. Protocol integrity checks

These checks are deliberately strict.

They verify that:

- E1 has both classes;
- calibration is separate from E1 evaluation;
- the final test has never been used to construct the stream;
- threshold values are finite;
- continual matrices contain no future-experience evaluation before the experience is learned.


In [24]:
# 24. Integrity checks

assert E1_y_train.min() == 0 and E1_y_train.max() == 1
assert E1_y_cal.min() == 0 and E1_y_cal.max() == 1
assert E1_y_eval.min() == 0 and E1_y_eval.max() == 1

for t in [
    static_threshold,
    naive_threshold,
    replay_threshold,
    ewc_threshold
]:
    assert 0.0 < t < 1.0

# Ensure the final test was not used in stream construction.
assert len(test_df) == len(y_test)

# Naive/replay/EWC matrices must not contain a value for future experiences.
for matrix in [naive_matrix, replay_matrix, ewc_matrix]:
    for i, state in enumerate(matrix.index):
        future = list(stream_data.keys())[i+1:]
        for exp in future:
            assert pd.isna(matrix.loc[state, exp])

print("ALL PROTOCOL INTEGRITY CHECKS PASSED.")


ALL PROTOCOL INTEGRITY CHECKS PASSED.


# 25. Save artifacts

The outputs below are the ones needed for the research decision:

1. `experience_summary.csv`
2. `final_stream_retention.csv`
3. `continual_learning_aggregate_results.csv`
4. `forgetting_analysis.csv`
5. `resource_summary.csv`
6. `final_temporal_test_comparison.csv`
7. model-specific threshold-selection CSV files
8. continual-learning F1 matrices

Do not use the previous Phase-2B pilot results as final thesis evidence.


In [25]:
# 25. Save protocol and models

joblib.dump(
    static_model,
    ARTIFACTS/"static_lightgbm.joblib"
)

joblib.dump(
    naive_model,
    ARTIFACTS/"naive_continual_lightgbm.joblib"
)

joblib.dump(
    replay_model,
    ARTIFACTS/"replay_lightgbm.joblib"
)

torch.save(
    ewc_model.state_dict(),
    ARTIFACTS/"ewc_mlp_state.pt"
)

protocol = {
    "dataset": "lacg030175/UNSW-NB15",
    "config": "temporal",
    "experiences": EXPERIENCES,
    "e1_split": {
        "training": 0.56,
        "calibration": 0.14,
        "evaluation": 0.30,
        "method": "stratified"
    },
    "later_experience_split": "first 70 percent adaptation, final 30 percent evaluation",
    "threshold_selection": {
        "source": "E1 calibration only",
        "fpr_constraint": 0.10,
        "range": [0.05, 0.95],
        "grid_points": 181,
        "frozen_after_e1": True
    },
    "final_test_used_for_training": False,
    "final_test_used_for_threshold_tuning": False,
    "methods": [
        "Static_LightGBM",
        "Naive_CL_LightGBM",
        "Replay_LightGBM",
        "EWC_MLP"
    ],
    "replay_per_experience": REPLAY_PER_EXPERIENCE,
    "ewc_lambda": EWC_LAMBDA,
    "ewc_epochs": EWC_EPOCHS,
    "seed": SEED,
    "important_note": (
        "E1 is stratified to avoid one-class initialization; "
        "therefore E1 itself is not treated as a strictly chronological micro-stream."
    )
}

with open(RESULTS/"phase2b_v2_protocol.json", "w") as f:
    json.dump(protocol, f, indent=2)

bundle = shutil.make_archive(
    str(BASE/"phase2b_v2_artifacts"),
    "zip",
    root_dir=BASE
)

print("Created:", bundle)
print("\nResult files:")
for p in sorted(RESULTS.glob("*")):
    print(" -", p)


Created: /content/carc_ids_phase2b_v2/phase2b_v2_artifacts.zip

Result files:
 - /content/carc_ids_phase2b_v2/results/continual_learning_aggregate_results.csv
 - /content/carc_ids_phase2b_v2/results/ewc_f1_matrix.csv
 - /content/carc_ids_phase2b_v2/results/ewc_threshold_selection.csv
 - /content/carc_ids_phase2b_v2/results/experience_summary.csv
 - /content/carc_ids_phase2b_v2/results/final_stream_retention.csv
 - /content/carc_ids_phase2b_v2/results/final_temporal_test_comparison.csv
 - /content/carc_ids_phase2b_v2/results/forgetting_analysis.csv
 - /content/carc_ids_phase2b_v2/results/naive_f1_matrix.csv
 - /content/carc_ids_phase2b_v2/results/naive_threshold_selection.csv
 - /content/carc_ids_phase2b_v2/results/phase2b_v2_protocol.json
 - /content/carc_ids_phase2b_v2/results/replay_f1_matrix.csv
 - /content/carc_ids_phase2b_v2/results/replay_threshold_selection.csv
 - /content/carc_ids_phase2b_v2/results/resource_summary.csv
 - /content/carc_ids_phase2b_v2/results/static_f1_by_exper

# 26. Decision gate

Do not proceed to the resource-aware controller until these results are inspected.

### The benchmark passes the gate if:

1. all methods start from a valid two-class E1 detector;
2. static baseline has sensible non-zero detection;
3. continual methods show interpretable adaptation/forgetting behavior;
4. final-test performance is plausible;
5. no leakage/integrity check fails.

### If replay already gives near-optimal performance at very low cost:

The proposed controller must demonstrate a meaningful advantage in **selective adaptation/resource utility**.

### If continual learning does not improve the security-cost trade-off:

We should not force a continual-learning contribution.

That is a research decision, not a coding failure.
